# Exercise on Model selection and shrinkage

Response variable is sleep duration (Sleep_Duration)

The data set contains several descriptors

Build a model that predict sleep duration from the descriptors. Use at least one model selection sechnique from lecture to pass.

Build a predictive model using shrinkage methods.



In [19]:
# Setup
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error # calculates mean squared error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline # used for standardizing predictors
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # used for standardizing predictors
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
import statsmodels.api as sm # used for ACI

## 1. Download data
You have to fix your own access token on the kaggle website, click in where they ask you if it's your first time using API on kaggle. the path = kagglehub.... line only needs to be run one time to put the datafiles in a folder

In [2]:
# Download data files and save in data folder (only done once for setup)
# path = kagglehub.competition_download('bern-02-2026', output_dir="./data")

In [3]:
# Use this to train and validate model (using kfold approach)
train_df = pd.read_csv("./data/train.csv")
train_df.head()
#print(train_df.shape)

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep,Sleep_Duration
0,311,Female,49,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-1.278341,0.440786,5.407088,7.474116,6.2
1,207,Male,30,Doctor,75,6,Normal,70,8000,NaN,-0.826910,0.691399,5.839617,7.979600,7.7
2,169,Male,35,Engineer,60,4,Normal,65,5000,NaN,1.139555,0.909316,7.039570,5.902686,7.3
3,149,Male,39,Lawyer,60,5,Normal,68,8000,NaN,2.069801,0.741677,4.464130,8.127826,7.2
4,98,Female,50,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-2.907425,0.418233,3.068291,8.099139,6.0


In [4]:
# Test data does not contain the Sleep_Duration
test_df = pd.read_csv("./data/test.csv")
test_df.head()

,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep
0,3,Male,31,Doctor,30,8,Normal,72,5000,NaN,-1.627562,0.474605,6.029331,5.970977
1,234,Male,43,Salesperson,45,7,Overweight,72,6000,Sleep Apnea,3.311945,0.567641,5.366240,5.824984
2,187,Female,44,Teacher,45,4,Overweight,65,6000,Insomnia,0.800264,0.924036,6.645139,7.465405
3,220,Male,33,Doctor,30,8,Normal,72,5000,NaN,4.397880,0.512827,8.466199,7.946357
4,106,Male,30,Doctor,75,6,Normal,70,8000,NaN,-0.383691,0.634333,6.751696,6.343607


In [5]:
# Use this as the format to save predictions
sample_submission = pd.read_csv("./data/sample_submission.csv")
sample_submission

,ID,Sleep_Duration
0,1,0.3
1,2,1.1
2,3,4.7


## 2. AIC
Next step is to try to use AIC to further improve on the model (and the predictions it makes). 

AIC balances fit and complexity by penalizing more estimated parameters. 
$$
AIC=-2\log(\hat L)+2k
$$
where $k$ is the number of estimated parameters and $\hat L$ is the likelihood. A lower AIC is preferred. 

### 2.1 Model selection using AIC

I use AIC to investigate whether a simpler linear regression model can describe the data without unnecessary predictors. The model selection is performed step by step:

**1. Fit the full model**  
Start with all available predictors and calculate its AIC.

**2. Remove one predictor at a time**  
Create separate reduced models by removing one predictor from the full model. 

**3. Compare AIC values**  
Calculate the AIC for each reduced model. 

- Lower AIC → better balance between model fit and complexity
- Higher AIC → the removed predictor may still be useful

**4. Keep the model with the lowest AIC**  

**5. Repeat**  
Starting from the new model, again test removing one predictor at a time.

**Stop when removing predictors no longer improves AIC**  
The resulting model is the AIC-selected model.

In [ ]:
def calculate_aic(df, predictors):
    X = df[predictors]
    
    # Convert categorical variables to dummy variables
    X = pd.get_dummies(X, drop_first=True)
    
    # Add intercept
    X = sm.add_constant(X)
    
    # Fit ordinary linear regression
    model = sm.OLS(df["Sleep_Duration"], X).fit()
    return model.aic

predictors = train_df.drop(columns=["Sleep_Duration", "ID"]).columns.tolist()
current_predictors = predictors.copy()
results = {}

# AIC of the full model
current_aic = calculate_aic(train_df, current_predictors)
print(f"Starting AIC: {current_aic:.2f}")

for step in range(len(predictors)):
    step_results = {}

    # Try removing each remaining predictor
    for predictor in current_predictors:
        reduced_predictors = [p for p in current_predictors if p != predictor]
        aic = calculate_aic(train_df, reduced_predictors)
        step_results[predictor] = aic

    # Find the removal giving the lowest AIC
    predictor_to_remove = min(step_results, key=step_results.get)
    best_aic = step_results[predictor_to_remove]
    print(f"Step {step + 1}: "f"Remove {predictor_to_remove} → AIC = {best_aic:.2f}")

    # Stop if AIC does not improve
    if best_aic >= current_aic:
        print("AIC no longer improves → stopping.")
        break

    # Keep the improvement
    current_predictors.remove(predictor_to_remove)
    current_aic = best_aic

    results[step] = {
        "removed": predictor_to_remove,
        "aic": current_aic,
        "predictors": current_predictors.copy()}

print("\nFinal predictors:")
print(current_predictors) # Use the current predictors to optimize this model
print(f"\nFinal AIC: {current_aic:.2f}")

Starting AIC: 70.01
Step 1: Remove Sleep_Disorder → AIC = 68.07
Step 2: Remove Mood → AIC = 66.20
Step 3: Remove Gender → AIC = 64.62
Step 4: Remove Quality_of_Sleep → AIC = 63.78
Step 5: Remove Daily_Steps → AIC = 63.50
Step 6: Remove Temp → AIC = 63.19
Step 7: Remove Physical_Activity_Level → AIC = 76.10
AIC no longer improves → stopping.

Final predictors:
['Age', 'Occupation', 'Physical_Activity_Level', 'Stress_Level', 'BMI_Category', 'Heart_Rate', 'History']

Final AIC: 63.19


Now we have a model with fewer predictors. Next step is to use a ridge approach to find the best parameters for these predictors


## 3. Fit some models - find the best $\alpha$
First, I define some functions that prepare the data, fit Ridge models, and calculate their MSE. I then run the model testing in the next cell.

I use **k-fold cross-validation** to evaluate the models on several different train/test splits instead of relying on just one split. This gives us a more reliable estimate of the model's performance and reduces the influence of any particular random split.

For each fold, I use **Ridge regression with different values of $\alpha$**. The $\alpha$ value controls the strength of the shrinkage: a larger $\alpha$ means stronger shrinkage of the regression coefficients towards zero. (larger $\alpha$ will more heavily penalice large coefficients in the model ($\beta_i$)). 

I calculate the test MSE for each $\alpha$ in each fold and then calculate the **mean MSE across the folds**. I use this mean MSE to determine which $\alpha$ gives the lowest prediction error.

In [71]:
# Prepare x and y
def separate_x_y(df): 
    # predictor columns (p = 13)
    x = df[[
        'Gender', 'Age', 'Occupation', 'Physical_Activity_Level',
        'Stress_Level', 'BMI_Category', 'Heart_Rate', 'Daily_Steps',
        'Sleep_Disorder', 'Temp', 'Mood', 'History', 'Quality_of_Sleep'
        ]]

    # response
    if "Sleep_Duration" in df.columns: # (there is no Sleep_Duration column in test_df) 
        y = df["Sleep_Duration"] 
        return x, y
    else:
        return x

# Try different alpha values and calculate the MSE using the ridge model
def evaluate_shrinkage(x_train, x_test, y_train, y_test, alphas, predictors):
    ridge_mse = []
    models = []

    for alpha in alphas:
        model = create_model(alpha, predictors)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        mse = mean_squared_error(y_test, y_pred)
                             
        ridge_mse.append(mse)
        models.append(model)

    return ridge_mse, models

# Create a model using Ridge Shrinkage and one alpha
def create_model(alpha, predictors):
    categorical_columns = ['Gender', 'Occupation', 'BMI_Category', 'Sleep_Disorder']
    
    numerical_columns = [
        'Age', 'Physical_Activity_Level', 'Stress_Level', 'Heart_Rate', 
        'Daily_Steps', 'Temp', 'Quality_of_Sleep', 'Mood', 'History']

    # If no predictors are specified, use all predictors
    if predictors is not None:
        # Only keep columns that are actually being used
        categorical_columns = [col for col in categorical_columns if col in predictors]
        numerical_columns = [col for col in numerical_columns if col in predictors]

    preprocessor = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), # add drop = "first", if i want one reference
            categorical_columns),                               # category for each categorical variable
        ("numerical", StandardScaler(),
            numerical_columns)
        ])
    
    model = Pipeline([
                ("preprocessor", preprocessor),
                ("ridge", Ridge(alpha=alpha))
            ])
    
    return model

# Implement the k fold split of the dataset
def k_fold_cv(df, kf, alphas, predictors = None):
    mse_test_list = []
    models_list = []

    for train_index, test_index in kf.split(df):
        # Make train and test datasets
        train = df.iloc[train_index]
        test = df.iloc[test_index]

        # Separate response and predictors
        x_train, y_train = separate_x_y(train)
        x_test, y_test = separate_x_y(test)

        # Keep only the predictors we want to use
        if predictors is None:   # if predictors = None: use all columns in x
            predictors = x_train.columns.tolist()
        x_train = x_train[predictors]
        x_test = x_test[predictors]

        # Evaluate Ridge MSE for all alpha values
        mse_test, models = evaluate_shrinkage(x_train, x_test, y_train, y_test, alphas, predictors)
        mse_test_list.append(mse_test); models_list.append(models)

    return mse_test_list, models_list

In [72]:
# Run model testing
k = 5
#alphas = [0.01, 0.1, 1, 10, 100]
alphas = [1, 5, 10, 15]

# Create 3 train/test splits of the data
kf = KFold(n_splits=k, shuffle=True, random_state=0)

mse_test_list, models_list = k_fold_cv(train_df, kf, alphas, current_predictors)

In [73]:
# Print results in a nice table

# Convert to array so we can calculate mean and SE for each alpha
mse_array = np.array(mse_test_list)
mean_mse = np.mean(mse_array, axis=0)
se_mse = np.std(mse_array, axis=0, ddof=1) / np.sqrt(k)

# Print results in a nice table
print("Test MSE by fold")
print("-" * 75)

print(f"{'Fold':<8}", end="")
for alpha in alphas:
    print(f"{'alpha=' + str(alpha):>12}", end="")
print()
print("-" * 75)

for fold, mse_values in enumerate(mse_test_list, start=1):
    print(f"{fold:<8}", end="")
    for mse in mse_values:
        print(f"{mse:>12.4f}", end="")
    print()
print("-" * 75)

# Mean MSE
print(f"{'Mean':<8}", end="")
for mse in mean_mse:
    print(f"{mse:>12.4f}", end="")
print()

# Standard error
print(f"{'SE':<8}", end="")
for se in se_mse:
    print(f"{se:>12.4f}", end="")
print()

print("\n(obs: remember that smaller MSE is better.)")

Test MSE by fold
---------------------------------------------------------------------------
Fold         alpha=1     alpha=5    alpha=10    alpha=15
---------------------------------------------------------------------------
1             0.1168      0.0983      0.0998      0.1033
2             0.0884      0.0965      0.1078      0.1172
3             0.0966      0.0912      0.0931      0.0976
4             0.0985      0.1143      0.1261      0.1359
5             0.0764      0.0657      0.0612      0.0616
---------------------------------------------------------------------------
Mean          0.0954      0.0932      0.0976      0.1031
SE            0.0066      0.0079      0.0107      0.0123

(obs: remember that smaller MSE is better.)


Continue with the $\alpha$ that created the lowest MSE. For now, I will use $\alpha = 5$

### 3.1 Look at generated models

In [28]:
# Get info on one specific ridge model generated from previous code blocks (alpha = 5)
models_list[0][1]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['Gender','Age','Occupation',...,'Mood','History','Quality_of_Sleep']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subse

In [29]:
# Here are some ways of printing the coefficients for the different models that were created
# Not useful exept for getting a visual understanding of what is happening

# get the coefficients from one of the models
model = models_list[0][1]
ridge = model.named_steps["ridge"]
print(ridge.coef_)

"""preprocessor = model.named_steps["preprocessor"]
feature_names = preprocessor.get_feature_names_out()
for name, coefficient in zip(feature_names, ridge.coef_):
    print(f"{name:<40} {coefficient:.4f}")"""

# print the coefficients for different values of alpha
"""for alpha, model in zip(alphas, models_list[0]):
    ridge = model.named_steps["ridge"]
    
    print(f"\nAlpha = {alpha}")
    print(ridge.coef_)"""


[-0.06255642  0.06255642 -0.07856747  0.23204664  0.10820262  0.02115358
 -0.05343574  0.01098939  0.08035732 -0.00709042 -0.05525385 -0.25840206
  0.12412853  0.1940205  -0.15590235 -0.16224668 -0.12115925  0.03620731
  0.08495194  0.17882816  0.17014561 -0.44369645  0.02090367 -0.09150134
  0.00472272  0.03334191  0.05638925  0.12187005]


'for alpha, model in zip(alphas, models_list[0]):\n    ridge = model.named_steps["ridge"]\n\n    print(f"\nAlpha = {alpha}")\n    print(ridge.coef_)'

## 4. Test the final models on the test data
I have selected $\alpha = 5$ based on the lowest mean cross-validation MSE. I. e. the performance metric I have used is mean squared error (MSE). The next step is to fit the final linear model using $\alpha = 5$ and all 180 observations in train_df.

The final model that was fitted:
$$
\text{Sleep Duration} =
\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_{13}x_{13}
$$

The ridge model used to calculate MSE:
$$
\text{MSE}+\alpha\sum_j\beta_j^2
$$

In [31]:
ridge = ridge_model.named_steps["ridge"]
preprocessor = ridge_model.named_steps["preprocessor"]

feature_names = preprocessor.get_feature_names_out()

# print coefficients for the final_model
"""for name, coefficient in zip(feature_names, ridge.coef_):
    print(f"{name:<40} {coefficient:.4f}")"""

'for name, coefficient in zip(feature_names, ridge.coef_):\n    print(f"{name:<40} {coefficient:.4f}")'

In [ ]:
# AIC + ridge

# Create model
ridge_model = create_model(alpha = 5, predictors=current_predictors)

# Fit on all of the training data
x, y = separate_x_y(train_df)
ridge_model.fit(x, y)

x_test = separate_x_y(test_df) # there is no y given for the test dataset
y_pred = ridge_model.predict(x_test) # make predictions

# Create the submission dataframe
predictions = pd.DataFrame({
    "ID": test_df["ID"],
    "Sleep_Duration": y_pred
    })

# Save as CSV to the data folder. Has the correct format for upploading the file to the kaggle competition
predictions.to_csv("./data/predictions.csv", index=False)

In [ ]:
# The actual best model is still this one (using all predictors and ridge model):

# Create model
ridge_model = create_model(alpha = 5, predictors=None)

# Fit on all of the training data
x, y = separate_x_y(train_df)
ridge_model.fit(x, y)
x_test = separate_x_y(test_df) # there is no y given for the test dataset
y_pred = ridge_model.predict(x_test) # make predictions

# Create the submission dataframe
predictions = pd.DataFrame({
    "ID": test_df["ID"],
    "Sleep_Duration": y_pred})

# Save as CSV to the data folder. Has the correct format for upploading the file to the kaggle competition
predictions.to_csv("./data/predictions.csv", index=False)